# ⚙️ 05 — Feature Engineering
## Silver · Turbina Kelmarsh T1 · 2018–2021

---

### Contexto

El dataset etiquetado contiene valores brutos de sensores a intervalos de 10 minutos. Los valores brutos tienen escasa capacidad predictiva porque un sensor no falla por un valor puntual alto, sino por patrones sostenidos en el tiempo.

### Diseño de features

Para cada sensor se calculan cuatro estadísticos en cuatro ventanas temporales:

| Estadístico | Qué captura |
|------------|-------------|
| `mean` | Nivel promedio — detecta derivas lentas |
| `std` | Variabilidad — detecta inestabilidad creciente |
| `p95` | Percentil 95 — el sensor empieza a tocar valores extremos |
| `exceedance` | Fracción del tiempo por encima del p90 del baseline |

Más un `baseline_ratio` en ventana 7d: media actual dividida entre la media de los primeros 180 días.

### Features de contexto temporal

Un problema clásico en mantenimiento predictivo con múltiples fallos por año es que los períodos de operación normal quedan contaminados: si hay un fallo cada 2-3 semanas, casi no existe ningún momento que esté lejos de todos los fallos. El modelo no puede distinguir 'período limpio tras reparación' de 'aproximándose al siguiente fallo'.

La solución es darle al modelo **contexto temporal explícito** mediante dos features:

| Feature | Descripción | Por qué ayuda |
|---------|-------------|---------------|
| `hours_since_last_fault` | Horas desde el último fallo de la familia | Tras una reparación la turbina está en estado limpio — prob baja aunque los sensores sean similares |
| `hours_since_last_fault_log` | Log(1 + horas_desde_último_fallo) | Comprime la escala larga, más fácil de aprender para árboles |

Con estas features el modelo aprende: 'acabo de reparar = negativo aunque el sensor esté alto' vs 'han pasado 3 semanas desde la reparación y el sensor está subiendo = positivo'.

### Por qué no se usan features de pendiente (slope)

Las features de pendiente mostraron AUC univariante = 0.500 en todas las familias. La degradación en turbinas eólicas no sigue una pendiente monótona. Se descartaron completamente.

## 1. Carga del Dataset Etiquetado

In [8]:
import os, time
import pandas as pd
import numpy as np

base_dir   = os.path.dirname(os.getcwd())
silver_dir = os.path.join(base_dir, 'data', 'silver')

df = pd.read_parquet(os.path.join(silver_dir, 'dataset_labeled.parquet'))
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset cargado: {len(df):,} filas × {len(df.columns)} columnas')
print(f'Rango: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print()
for fam in ['yaw_cable', 'brake_hydro', 'generator', 'pitch_bat']:
    n = df[f'is_pre_{fam}'].sum()
    print(f'  {fam:<15}: {n:>7,} positivos ({100*n/len(df):.1f}%)')


Dataset cargado: 210,384 filas × 55 columnas
Rango: 2017-12-31 → 2021-12-31

  yaw_cable      :  52,639 positivos (25.0%)
  brake_hydro    :  21,185 positivos (10.1%)
  generator      :  22,272 positivos (10.6%)
  pitch_bat      :  28,753 positivos (13.7%)


---

## 2. Features de Dominio

In [9]:
original_columns = set(df.columns)

# --- YAW / CABLE ---
df['yaw_error']        = (df['nacelle_position'] - df['wind_direction']).abs() % 360
df['yaw_error']        = df['yaw_error'].apply(lambda x: x if x <= 180 else 360 - x)
df['yaw_error_wind']   = df['yaw_error'] * df['wind_speed_ms']
df['cable_rate']       = df['cable_windings_from_calibration_point'].diff(1).fillna(0)
df['nacelle_std_ratio']= df['nacelle_position_standard_deviation'] / (df['wind_speed_ms'] + 1e-6)

# --- GENERADOR (térmicas) ---
df['t_bearing_delta']      = df['generator_bearing_front_temperature_c'] - df['nacelle_ambient_temperature_c']
df['t_rear_bearing_delta'] = df['generator_bearing_rear_temperature_c']  - df['nacelle_ambient_temperature_c']
df['t_stator_delta']       = df['stator_temperature_1_c']                - df['nacelle_ambient_temperature_c']
df['t_gear_oil_delta']     = df['gear_oil_temperature_c']                - df['nacelle_ambient_temperature_c']
df['t_bearing_diff']       = df['generator_bearing_front_temperature_c'] - df['generator_bearing_rear_temperature_c']
df['t_stator_bearing_diff']= df['stator_temperature_1_c']                - df['generator_bearing_front_temperature_c']
df['t_bearing_delta_roc']  = df['t_bearing_delta'].diff(6)
df['t_stator_roc']         = df['stator_temperature_1_c'].diff(6)

# --- ELÉCTRICO ---
df['apparent_power_kva']   = (df['power_kw']**2 + df['reactive_power_kvar']**2) ** 0.5
df['reactive_power_ratio'] = df['reactive_power_kvar'] / (df['apparent_power_kva'] + 1e-6)

# --- HIDRÁULICO ---
df['pressure_vs_temp']    = df['gear_oil_inlet_pressure_bar'] / (df['gear_oil_inlet_temperature_c'] + 273.15)
df['metal_particle_rate'] = df['metal_particle_count'].diff(1).fillna(0).clip(lower=0)

# --- PITCH / BATERÍAS ---
df['t_motor1_vs_ambient'] = df['temperature_motor_axis_1_c'] - df['nacelle_ambient_temperature_c']
df['t_motor2_vs_ambient'] = df['temperature_motor_axis_2_c'] - df['nacelle_ambient_temperature_c']
df['t_motor3_vs_ambient'] = df['temperature_motor_axis_3_c'] - df['nacelle_ambient_temperature_c']
df['pitch_asymmetry']     = (df[['blade_angle_pitch_position_a',
                                  'blade_angle_pitch_position_b',
                                  'blade_angle_pitch_position_c']].max(axis=1) -
                              df[['blade_angle_pitch_position_a',
                                  'blade_angle_pitch_position_b',
                                  'blade_angle_pitch_position_c']].min(axis=1))
df['blade_angle_mean']        = df[['blade_angle_pitch_position_a',
                                     'blade_angle_pitch_position_b',
                                     'blade_angle_pitch_position_c']].mean(axis=1)
df['motor_current_imbalance'] = df[['motor_current_axis_1_a',
                                     'motor_current_axis_2_a',
                                     'motor_current_axis_3_a']].std(axis=1)

calc = [c for c in df.columns if c not in original_columns]
print(f'Features de dominio calculadas: {len(calc)}')


Features de dominio calculadas: 22


---

## 3. Features de Contexto Temporal

Estas features son las más críticas para resolver el problema de precisión.

Sin contexto temporal, el modelo ve sensores en estado similar tanto justo después de una reparación (negativo real) como semanas antes del siguiente fallo (positivo real). Con `hours_since_last_fault`, el modelo aprende que el período inmediatamente posterior a una reparación es de bajo riesgo, aunque los sensores no hayan vuelto del todo a la línea base.

In [10]:
# ==============================================================================
# Features de contexto temporal por familia
# ==============================================================================
targets_df = pd.read_parquet(os.path.join(silver_dir, 'fault_targets_grouped.parquet'))

def add_temporal_context(df, family, fault_targets):
    fault_times = sorted(
        fault_targets[fault_targets['family'] == family]['timestamp'].tolist()
    )
    fault_arr = np.array(fault_times, dtype='datetime64[ns]')
    ts_arr    = df['timestamp'].values.astype('datetime64[ns]')

    hours_since = np.full(len(ts_arr), np.nan)
    for i, ts in enumerate(ts_arr):
        past = fault_arr[fault_arr <= ts]
        if len(past) == 0:
            # Antes del primer fallo histórico: asignar valor alto (operación nueva)
            hours_since[i] = 8760.0  # 1 año
        else:
            hours_since[i] = (ts - past[-1]) / np.timedelta64(1, 'h')

    df[f'hours_since_last_{family}']     = hours_since
    # Versión log: comprime la escala para facilitar el aprendizaje de árboles
    df[f'hours_since_last_{family}_log'] = np.log1p(hours_since)
    return df


print('Calculando contexto temporal...')
for family in ['yaw_cable', 'brake_hydro', 'generator', 'pitch_bat']:
    t0 = time.time()
    df = add_temporal_context(df, family, targets_df)
    # Verificación: los positivos deben tener hours_since más altas que los negativos cercanos
    pos_mean = df.loc[df[f'is_pre_{family}'],  f'hours_since_last_{family}'].mean()
    neg_mean = df.loc[~df[f'is_pre_{family}'], f'hours_since_last_{family}'].mean()
    print(f'  {family:<15}: pos_mean={pos_mean:>7.1f}h  neg_mean={neg_mean:>7.1f}h  [{time.time()-t0:.1f}s]')

print()
print('Interpretación: pos_mean > neg_mean confirma que los fallos ocurren más lejos')
print('de la última reparación que los períodos normales inmediatos post-reparación.')


Calculando contexto temporal...
  yaw_cable      : pos_mean=  306.0h  neg_mean=  801.8h  [0.7s]
  brake_hydro    : pos_mean= 1323.1h  neg_mean= 1154.2h  [0.7s]
  generator      : pos_mean= 1252.7h  neg_mean= 1345.1h  [0.7s]
  pitch_bat      : pos_mean= 2713.1h  neg_mean= 1971.0h  [0.7s]

Interpretación: pos_mean > neg_mean confirma que los fallos ocurren más lejos
de la última reparación que los períodos normales inmediatos post-reparación.


---

## 4. Baseline de Operación Limpia

In [11]:
BASELINE_DAYS   = 180
baseline_cutoff = df['timestamp'].min() + pd.Timedelta(days=BASELINE_DAYS)
df_baseline     = df[df['timestamp'] < baseline_cutoff]

sensor_cols = [
    c for c in df.columns
    if c not in ['timestamp']
    and not c.startswith('is_pre_')
    and not c.startswith('hours_to_')
    and not c.startswith('hours_since_')
    and df[c].dtype in [float, 'float64', 'float32']
]
baseline_mean = df_baseline[sensor_cols].mean()
baseline_p90  = df_baseline[sensor_cols].quantile(0.90)

print(f'Período baseline: {df["timestamp"].min().date()} → {baseline_cutoff.date()}')
print(f'Sensores cubiertos: {len(sensor_cols)}')


Período baseline: 2017-12-31 → 2018-06-29
Sensores cubiertos: 67


---

## 5. Función de Rolling Features

In [12]:
WINDOWS = {'1h': 6, '6h': 36, '24h': 144, '7d': 1008}

def make_rolling_features(df, sensors, windows, baseline_mean, baseline_p90):
    feats = {}
    for col in sensors:
        if col not in df.columns:
            print(f'  WARN: {col} no encontrada')
            continue
        s = df[col].ffill().fillna(0)
        thresh = baseline_p90.get(col, s.quantile(0.90))

        for wname, w in windows.items():
            mp   = max(1, w // 3)
            roll = s.rolling(w, min_periods=mp)
            feats[f'{col}__mean_{wname}']   = roll.mean()
            feats[f'{col}__std_{wname}']    = roll.std().fillna(0)
            feats[f'{col}__p95_{wname}']    = roll.quantile(0.95)
            feats[f'{col}__exceed_{wname}'] = s.rolling(w, min_periods=mp).apply(
                lambda x: (x > thresh).mean(), raw=True)

        bm = baseline_mean.get(col, 1.0)
        if abs(bm) > 1e-6:
            roll_7d = s.rolling(WINDOWS['7d'], min_periods=max(1, WINDOWS['7d'] // 3))
            feats[f'{col}__baseline_ratio'] = roll_7d.mean() / abs(bm)

    return pd.DataFrame(feats, index=df.index)

print('Función definida')


Función definida


---

## 6. Sensores por Familia

In [13]:
FAMILY_SENSORS = {

    'yaw_cable': [
        'nacelle_position', 'nacelle_position_standard_deviation',
        'wind_direction', 'wind_direction_standard_deviation',
        'vane_position_12', 'cable_windings_from_calibration_point',
        'wind_speed_ms', 'power_kw',
        'yaw_error', 'yaw_error_wind', 'cable_rate', 'nacelle_std_ratio',
    ],

    'brake_hydro': [
        'gear_oil_inlet_pressure_bar', 'gear_oil_pump_pressure_bar',
        'gear_oil_inlet_temperature_c', 'gear_oil_temperature_c',
        'generator_rpm_rpm', 'generator_rpm_standard_deviation_rpm',
        'rotor_speed_rpm', 'power_kw',
        'front_bearing_temperature_c', 'rear_bearing_temperature_c',
        'metal_particle_count',
        't_gear_oil_delta', 'pressure_vs_temp', 'metal_particle_rate',
    ],

    'generator': [
        'generator_bearing_front_temperature_c', 'generator_bearing_rear_temperature_c',
        'generator_bearing_front_temperature_max_c', 'generator_bearing_rear_temperature_max_c',
        'nacelle_temperature_c', 'nacelle_ambient_temperature_c',
        'ambient_temperature_converter_c', 'power_kw', 'reactive_power_kvar',
        'power_factor_cosphi', 'stator_temperature_1_c', 'wind_speed_ms',
        't_bearing_delta', 't_rear_bearing_delta', 't_stator_delta',
        't_bearing_diff', 't_stator_bearing_diff',
        'apparent_power_kva', 'reactive_power_ratio',
        't_bearing_delta_roc', 't_stator_roc',
    ],

    # NOTA: se usan deltas motor-ambiente en lugar de temperatura absoluta.
    # Con temperatura absoluta el modelo aprende estacionalidad en lugar de
    # degradación de batería.
    'pitch_bat': [
        'motor_current_axis_1_a', 'motor_current_axis_2_a', 'motor_current_axis_3_a',
        'blade_angle_pitch_position_a', 'blade_angle_pitch_position_b', 'blade_angle_pitch_position_c',
        't_motor1_vs_ambient', 't_motor2_vs_ambient', 't_motor3_vs_ambient',
        'power_kw', 'wind_speed_ms',
        'pitch_asymmetry', 'blade_angle_mean', 'motor_current_imbalance',
    ],
}

n_feats_per_sensor = len(WINDOWS) * 4 + 1
for fam, sensors in FAMILY_SENSORS.items():
    # +2 por hours_since y hours_since_log
    total = len(sensors) * n_feats_per_sensor + 2
    print(f'{fam:<15}: {len(sensors)} sensores → ~{total} features (incl. contexto temporal)')


yaw_cable      : 12 sensores → ~206 features (incl. contexto temporal)
brake_hydro    : 14 sensores → ~240 features (incl. contexto temporal)
generator      : 21 sensores → ~359 features (incl. contexto temporal)
pitch_bat      : 14 sensores → ~240 features (incl. contexto temporal)


---

## 7. Generación y Exportación

> ⚠️ Este proceso tarda **10–20 minutos** por el rolling de 7d.

Las features `hours_since_last_{familia}` y `hours_since_last_{familia}_log` se añaden al final de cada archivo para que el modelo las tenga siempre disponibles.

In [14]:
for family, sensors in FAMILY_SENSORS.items():
    t0 = time.time()
    print(f'\n{"+"*60}')
    print(f'  Generando features: {family}')
    print(f'{"+"*60}')

    feats  = make_rolling_features(df, sensors, WINDOWS, baseline_mean, baseline_p90)

    # Añadir features de contexto temporal
    feats[f'hours_since_last_{family}']     = df[f'hours_since_last_{family}'].values
    feats[f'hours_since_last_{family}_log'] = df[f'hours_since_last_{family}_log'].values

    tcols  = [f'hours_to_{family}', f'is_pre_{family}']
    output = pd.concat([df[['timestamp']], df[tcols], feats], axis=1)

    out_path = os.path.join(silver_dir, f'features_{family}.parquet')
    output.to_parquet(out_path, index=False)

    n_nan = feats.isna().sum().sum()
    print(f'  ✅ features_{family}.parquet')
    print(f'     Shape:     {output.shape}')
    print(f'     Features:  {feats.shape[1]}')
    print(f'     Positivos: {output[f"is_pre_{family}"].sum():,} ({100*output[f"is_pre_{family}"].mean():.1f}%)')
    print(f'     NaN:       {n_nan:,} ({100*n_nan/feats.size:.2f}%)')
    print(f'     Tiempo:    {time.time()-t0:.0f}s')

print(f'\n✅ FEATURE ENGINEERING COMPLETADO')



++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: yaw_cable
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_yaw_cable.parquet
     Shape:     (210384, 209)
     Features:  206
     Positivos: 52,639 (25.0%)
     NaN:       18,204 (0.04%)
     Tiempo:    34s

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: brake_hydro
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_brake_hydro.parquet
     Shape:     (210384, 243)
     Features:  240
     Positivos: 21,185 (10.1%)
     NaN:       21,238 (0.04%)
     Tiempo:    39s

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Generando features: generator
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  ✅ features_generator.parquet
     Shape:     (210384, 362)
     Features:  359
     Positivos: 22,272 (10.6%)
     NaN:       31,857 (0.04%)
     Tiempo:    59s

+++++++++++++++++++++++++++++++++++++++++

---

## 📋 Conclusiones

Se han generado 4 archivos `features_{familia}.parquet` con las features rolling más las dos features de contexto temporal por familia.

**Las features `hours_since_last_fault` son las más importantes del pipeline**: permiten al modelo distinguir entre 'período limpio post-reparación' y 'aproximándose al siguiente fallo', que es precisamente la fuente de los falsos positivos masivos observados sin esta feature.

**Siguiente paso:** notebooks `06_train_{familia}.ipynb`.